<a href="https://colab.research.google.com/github/SpringBoard0811/Unified-Military-Analytics-and-Comparison/blob/Unified-Military-Analysis_Kavya/Military_Analysis_Milestone1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MILESTONE 1

# Task 3: Scraper

We install and load all the libraries needed to scrape, parse and store data, and set up browser-like headers so GFP doesn't block our requests.

In [ ]:
#cell 1 install & imports
#install
!pip install requests beautifulsoup4 pandas tqdm --quiet

# imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re, time, random, os
from tqdm.notebook import tqdm

In [ ]:
#mentioning headers to make GFP feel like we are a browser trying to get access

BASE = "https://www.globalfirepower.com"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": BASE,
}

session = requests.Session()
print("✅  Ready.")


✅  Ready.


requests fetches web pages, BeautifulSoup reads the HTML, pandas stores the data as a table, tqdm shows a progress bar, and session keeps our connection open across all requests.

# Task 4: Scraping using provided Links
We read the links file and extract each URL and its corresponding column name using regex.

In [ ]:
# parsing
LINKS_FILE = "links for military data.txt"   # must be uploaded to Colab

with open(LINKS_FILE, "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

print(f"Total lines in file: {len(raw_lines)}\n")

# Regex: capture the URL and the column name from each line
LINE_RE = re.compile(
    r"['\"]?(https?://[^\s'\"]+)['\"]?"   # URL (group 1)
    r"\s*[,:]\s*"                          # separator  : or ,
    r"['\"]([a-z_][a-z_0-9]*)['\"]"       # column name in quotes (group 2)
)

# Also capture the base_url separately
BASE_URL_RE = re.compile(r"base_url\s*[=:]\s*['\"]?(https?://[^\s'\"]+)")

sources = []    # list of (url, column_name)
base_url = None

for line in raw_lines:
    line = line.strip()
    if not line or line.startswith("#"):
        continue

    # Check for base_url
    m = BASE_URL_RE.search(line)
    if m:
        base_url = m.group(1).rstrip("'\"")
        print(f"  base_url → {base_url}")
        continue

    # Check for url: column_name pairs
    m = LINE_RE.search(line)
    if m:
        url = m.group(1).rstrip("',\"")
        col = m.group(2).strip()
        sources.append((url, col))

print(f"\n✅  Extracted {len(sources)} metric sources:")
for url, col in sources:
    print(f"   {col:45s} ← {url}")

if base_url:
    print(f"\n   Base URL (rankings page): {base_url}")

Total lines in file: 57

  base_url → https://www.globalfirepower.com/countries-listing.php

✅  Extracted 54 metric sources:
   total_population                              ← https://www.globalfirepower.com/total-population-by-country.php
   total_military_manpower                       ← https://www.globalfirepower.com/available-military-manpower.php
   fit_for_service                               ← https://www.globalfirepower.com/manpower-fit-for-military-service.php
   population_reaching_military_age_annually     ← https://www.globalfirepower.com/manpower-reaching-military-age-annually.php
   active_personnel                              ← https://www.globalfirepower.com/active-military-manpower.php
   reserve_personnel                             ← https://www.globalfirepower.com/active-reserve-military-manpower.php
   paramilitary                                  ← https://www.globalfirepower.com/manpower-paramilitary.php
   total_military_aircraft                       ← https

re.compile() builds a pattern to find text matching a specific format.
LINE_RE pulls out the URL and column name from each line.

Output should show metric sources successfully extracted.

We define two reusable functions that every scraping cell will call.

fetch to download a page & parse_num to clean a number.

In [ ]:
# fetching
def fetch(url: str, retries: int = 3) -> BeautifulSoup | None:
    """Fetch a URL and return a BeautifulSoup object, or None on failure."""
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, headers=HEADERS, timeout=20)
            r.raise_for_status()
            return BeautifulSoup(r.text, "html.parser")
        except requests.RequestException as e:
            print(f"  ⚠️  Attempt {attempt}/{retries} failed: {e}")
            if attempt < retries:
                time.sleep(2 * attempt)
    print(f"  ❌  Giving up on: {url}")
    return None


def parse_num(raw: str):
    """Parse a GFP numeric string → float, or None for N/A / missing."""
    if not raw:
        return None
    text = raw.strip()
    if text.upper() in ("N/A", "NA", "-", "--", "UNKNOWN", ""):
        return None
    text = text.replace(",", "").replace("$", "").replace("%", "").strip()
    # Handle shorthand: 1.5B, 300M, 50K
    m = re.match(r"^([\d.]+)\s*([KMBT])$", text, re.IGNORECASE)
    if m:
        mult = {"K": 1e3, "M": 1e6, "B": 1e9, "T": 1e12}[m.group(2).upper()]
        return float(m.group(1)) * mult
    try:
        return float(text)
    except ValueError:
        return None


fetch() downloads a URL and retries up to 3 times if it fails.

parse_num() converts raw strings like "2,035,000" or "N/A" into proper numbers, returning None for anything invalid so garbage values never enter the dataset.

We scrape the main GFP rankings page to get each country's rank, power index score, and unique ID.

In [ ]:
# scraping
RANKINGS_URL = base_url if base_url else f"{BASE}/countries-listing.php"

print(f"Fetching rankings from: {RANKINGS_URL}")
soup_rank = fetch(RANKINGS_URL)

rank_records = []

if soup_rank:
    # Every ranked country is an <a> tag linking to country-military-strength-detail.php
    links = soup_rank.find_all("a", href=re.compile(r"country-military-strength-detail\.php\?country_id="))

    for link in links:
        href = link.get("href", "")
        # Extract country_id slug
        m = re.search(r"country_id=([\w-]+)", href)
        if not m:
            continue
        country_id = m.group(1)

        text = link.get_text(" ", strip=True)

        # Extract rank (first number in text)
        rank_m = re.search(r"^(\d+)", text)
        rank = int(rank_m.group(1)) if rank_m else None

        # Extract PwrIndx score
        idx_m = re.search(r"PwrIndx[:\s]*([\d.]+)", text)
        power_index = float(idx_m.group(1)) if idx_m else None

        # Country name: text between rank and 3-letter code, stripping PwrIndx part
        # Pattern: "1 United States USA PwrIndx: 0.0741"
        name_m = re.search(r"^\d+\s+(.+?)\s+[A-Z]{2,3}\s+(?:PwrIndx|$)", text)
        if not name_m:
            # Fallback: title attribute on the link
            title = link.get("title", "")
            name_m2 = re.match(r"Military strength values of (.+?) for", title)
            country_name = name_m2.group(1).strip() if name_m2 else country_id.replace("-", " ").title()
        else:
            country_name = name_m.group(1).strip()

        if rank:
            rank_records.append({
                "country_id":        country_id,
                "country":           country_name,
                "power_index_rank":  rank,
                "power_index":       power_index,
            })

df_ranks = pd.DataFrame(rank_records).drop_duplicates("country_id").reset_index(drop=True)
print(f"\n✅  Rankings scraped: {len(df_ranks)} countries")
print(df_ranks[["power_index_rank", "country", "power_index"]].head(10).to_string(index=False))

Fetching rankings from: https://www.globalfirepower.com/countries-listing.php

✅  Rankings scraped: 145 countries
 power_index_rank        country  power_index
                1  United States       0.0741
                2         Russia       0.0791
                3          China       0.0919
                4          India       0.1346
                5    South Korea       0.1642
                6         France       0.1798
                7          Japan       0.1876
                8 United Kingdom       0.1881
                9        Turkiye       0.1975
               10          Italy       0.2211


This gives us our base table of 145 countries with their power_index_rank and power_index. The country is the key we'll use to join all other data onto this table later.

We scrape all category pages for the metrics that work normally (Part A), then visit each country's detail page individually for the metrics that failed (Part B), and finally merge everything into one wide table (Part C).

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re, time, random
from tqdm.notebook import tqdm

# ── These are the columns that FAILED with the link-based approach ────────────
# We'll scrape them directly from each country's detail page instead.
DETAIL_PAGE_COLS = {
    "oil_production_bbl",
    "oil_consumption_bbl",
    "proven_oil_reserves_bbl",
    "natural_gas_production_cum",
    "natural_gas_consumption_cum",
    "proven_natural_gas_reserves_cum",
    "coal_production_cum",
    "coal_consumption_mt",
    "proven_coal_reserves_cum",
    "total_land_area_sq_km",
    "coastline_coverage_km",
    "border_coverage_km",
    "waterway_coverage_km",
    "railway_coverage_km",
    "roadway_coverage_km",
}

# ── Patterns to extract each metric from a country detail page's raw text ─────
# Each pattern captures the numeric value for that metric.
# GFP format examples:
#   "Oil Production: 13,300,000 bbl/day"
#   "Railway Coverage: 93,141 km"
#   "Proven Coal Reserves: 247,883,000,000 mt"
DETAIL_PATTERNS = {
    "oil_production_bbl":              r"Oil\s*Production\s*[:\-]\s*([\d,]+)",
    "oil_consumption_bbl":             r"Oil\s*Consumption\s*[:\-]\s*([\d,]+)",
    "proven_oil_reserves_bbl":         r"Proven\s*Oil\s*Reserves\s*[:\-]\s*([\d,]+)",
    "natural_gas_production_cum":      r"(?:Natural\s*Gas\s*Production|NatGas\s*Production)\s*[:\-]\s*([\d,]+)",
    "natural_gas_consumption_cum":     r"(?:Natural\s*Gas\s*Consumption|NatGas\s*Consumption)\s*[:\-]\s*([\d,]+)",
    "proven_natural_gas_reserves_cum": r"Proven\s*Natural\s*Gas\s*Reserves\s*[:\-]\s*([\d,]+)",
    "coal_production_cum":             r"Coal\s*Production\s*[:\-]\s*([\d,]+)",
    "coal_consumption_mt":             r"Coal\s*Consumption\s*[:\-]\s*([\d,]+)",
    "proven_coal_reserves_cum":        r"(?:Coal\s*Proven\s*Reserves|Proven\s*Coal\s*Reserves)\s*[:\-]\s*([\d,]+)",
    "total_land_area_sq_km":           r"(?:Square\s*Land\s*Area|Total\s*Land\s*Area)\s*[:\-]\s*([\d,]+)",
    "coastline_coverage_km":           r"Coastline\s*Coverage\s*[:\-]\s*([\d,]+)",
    "border_coverage_km":              r"(?:Shared\s*Border|Border\s*Coverage)\s*[:\-]\s*([\d,]+)",
    "waterway_coverage_km":            r"Waterway\s*Coverage\s*[:\-]\s*([\d,]+)",
    "railway_coverage_km":             r"Railway\s*Coverage\s*[:\-]\s*([\d,]+)",
    "roadway_coverage_km":             r"Roadway\s*Coverage\s*[:\-]\s*([\d,]+)",
}


def parse_num(raw: str):
    """Parse a GFP numeric string → float, or None for N/A / missing."""
    if not raw:
        return None
    text = raw.strip()
    if text.upper() in ("N/A", "NA", "-", "--", "UNKNOWN", ""):
        return None
    text = text.replace(",", "").replace("$", "").replace("%", "").strip()
    m = re.match(r"^([\d.]+)\s*([KMBT])$", text, re.IGNORECASE)
    if m:
        mult = {"K": 1e3, "M": 1e6, "B": 1e9, "T": 1e12}[m.group(2).upper()]
        return float(m.group(1)) * mult
    try:
        return float(text)
    except ValueError:
        return None


HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.globalfirepower.com/",
}
BASE = "https://www.globalfirepower.com"
session = requests.Session()


def fetch(url: str, retries: int = 3):
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, headers=HEADERS, timeout=20)
            r.raise_for_status()
            return BeautifulSoup(r.text, "html.parser")
        except requests.RequestException as e:
            print(f"  ⚠️  Attempt {attempt}/{retries}: {e}")
            if attempt < retries:
                time.sleep(2 * attempt)
    return None


# ── PART A: Re-run link-based scraper for pages that work ────────────────────
# (This is the same logic as before, but now only for NON-failing columns)

# `sources` and `df_ranks` must already exist from Cells 3 & 5.
# If you're running this as a standalone replacement, re-run Cells 3-5 first.

metric_dfs = []

working_sources  = [(url, col) for url, col in sources if col not in DETAIL_PAGE_COLS]
failing_sources  = [(url, col) for url, col in sources if col in DETAIL_PAGE_COLS]

print(f"Link-based pages (working):  {len(working_sources)}")
print(f"Detail-page fallback needed: {len(failing_sources)}")
print(f"Failing columns: {[col for _, col in failing_sources]}\n")

print("── PART A: Scraping working category pages ──────────────────────────────")
for url, col_name in tqdm(working_sources, desc="Category pages"):
    soup = fetch(url)
    if soup is None:
        print(f"  ⚠️  Skipped: {col_name}")
        continue

    rows = []
    links = soup.find_all(
        "a",
        href=re.compile(r"country-military-strength-detail\.php\?country_id=")
    )

    for link in links:
        href = link.get("href", "")
        m = re.search(r"country_id=([\w-]+)", href)
        if not m:
            continue
        country_id = m.group(1)
        text = link.get_text(" ", strip=True)
        tokens = text.split()
        raw_value = tokens[-1] if tokens else ""
        value = parse_num(raw_value)
        if value is not None:
            rows.append({"country_id": country_id, col_name: value})

    if rows:
        df_metric = pd.DataFrame(rows).drop_duplicates("country_id")
        metric_dfs.append(df_metric)
        print(f"  ✅  {col_name:45s} → {len(rows)} values")
    else:
        print(f"  ❌  {col_name:45s} → still 0 (check that URL manually)")

    time.sleep(random.uniform(1.5, 2.5))


# ── PART B: Scrape country detail pages for the failing metrics ───────────────
# We visit each of the 145 country detail pages once and extract ALL
# the failing columns in a single pass — so it's still only 145 requests.

print("\n── PART B: Scraping country detail pages for resource/geography metrics ─")
print(f"  Columns to extract: {list(DETAIL_PAGE_COLS)}\n")

detail_records = []   # one dict per country

for _, row in tqdm(df_ranks.iterrows(), total=len(df_ranks), desc="Country detail pages"):
    country_id = row["country_id"]
    url = f"{BASE}/country-military-strength-detail.php?country_id={country_id}"

    soup = fetch(url)
    if soup is None:
        detail_records.append({"country_id": country_id})
        continue

    # Get all text from the page in one shot
    page_text = soup.get_text(" ", strip=True)

    record = {"country_id": country_id}

    for col, pattern in DETAIL_PATTERNS.items():
        m = re.search(pattern, page_text, re.IGNORECASE)
        if m:
            val = parse_num(m.group(1))
            if val is not None:
                record[col] = val

    detail_records.append(record)
    time.sleep(random.uniform(1.5, 2.5))

df_detail = pd.DataFrame(detail_records)
print(f"\n✅  Detail page scrape done: {len(df_detail)} countries")

# Show coverage for each previously-failing column
for col in DETAIL_PAGE_COLS:
    if col in df_detail.columns:
        filled = df_detail[col].notna().sum()
        print(f"  {col:45s} → {filled}/145 values extracted")
    else:
        print(f"  {col:45s} → column missing entirely")

metric_dfs.append(df_detail)


# ── PART C: Merge everything into final DataFrame ─────────────────────────────
print("\n── PART C: Merging all data ─────────────────────────────────────────────")

df = df_ranks.copy()
for df_metric in metric_dfs:
    df = df.merge(df_metric, on="country_id", how="left")

df = df.sort_values("power_index_rank").reset_index(drop=True)
print(f"\n✅  Final shape: {df.shape[0]} rows × {df.shape[1]} columns")

# Quick missing-value summary
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
problem_cols = missing_pct[missing_pct > 20].sort_values(ascending=False)
if len(problem_cols):
    print(f"\n⚠️  Columns still >20% missing:")
    print(problem_cols.to_string())
else:
    print("✅  All columns ≤20% missing — looking good!")

# Preview
key_cols = [c for c in [
    "power_index_rank", "country", "power_index",
    "active_personnel", "oil_production_bbl",
    "coastline_coverage_km", "railway_coverage_km",
    "proven_coal_reserves_cum",
] if c in df.columns]
print(f"\nSample (top 10):\n")
print(df[key_cols].head(10).to_string(index=False))


# ── Save & download ───────────────────────────────────────────────────────────
OUTPUT = "military_raw_data.csv"
df.to_csv(OUTPUT, index=False)
print(f"\n✅  Saved → {OUTPUT}  ({df.shape[0]} rows × {df.shape[1]} cols)")

from google.colab import files
files.download(OUTPUT)


Link-based pages (working):  39
Detail-page fallback needed: 15
Failing columns: ['railway_coverage_km', 'roadway_coverage_km', 'oil_production_bbl', 'oil_consumption_bbl', 'proven_oil_reserves_bbl', 'natural_gas_production_cum', 'natural_gas_consumption_cum', 'proven_natural_gas_reserves_cum', 'coal_production_cum', 'coal_consumption_mt', 'proven_coal_reserves_cum', 'total_land_area_sq_km', 'coastline_coverage_km', 'border_coverage_km', 'waterway_coverage_km']

── PART A: Scraping working category pages ──────────────────────────────


Category pages:   0%|          | 0/39 [00:00<?, ?it/s]

  ✅  total_population                              → 145 values
  ✅  total_military_manpower                       → 145 values
  ✅  fit_for_service                               → 145 values
  ✅  population_reaching_military_age_annually     → 145 values
  ✅  active_personnel                              → 145 values
  ✅  reserve_personnel                             → 145 values
  ✅  paramilitary                                  → 145 values
  ✅  total_military_aircraft                       → 145 values
  ✅  fighter_aircraft                              → 145 values
  ✅  attack_aircraft                               → 145 values
  ✅  transport_aircraft                            → 145 values
  ✅  trainer_aircraft                              → 145 values
  ✅  special_mission_aircraft                      → 145 values
  ✅  tanker_aircraft                               → 145 values
  ✅  total_military_helicopters                    → 145 values
  ✅  attack_helicopters                 

Country detail pages:   0%|          | 0/145 [00:00<?, ?it/s]


✅  Detail page scrape done: 145 countries
  oil_consumption_bbl                           → 145/145 values extracted
  coastline_coverage_km                         → 113/145 values extracted
  natural_gas_consumption_cum                   → 145/145 values extracted
  roadway_coverage_km                           → 145/145 values extracted
  total_land_area_sq_km                         → 145/145 values extracted
  oil_production_bbl                            → 145/145 values extracted
  border_coverage_km                            → column missing entirely
  proven_natural_gas_reserves_cum               → column missing entirely
  proven_oil_reserves_bbl                       → column missing entirely
  coal_consumption_mt                           → 145/145 values extracted
  waterway_coverage_km                          → column missing entirely
  coal_production_cum                           → 145/145 values extracted
  railway_coverage_km                           → 145/145 val

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Part A uses link-based extraction — each country on a category page is a clickable link containing the value.

Part B uses regex on raw page text to find values like "Oil Production: 13,300,000".

Part C uses df.merge() to join all small tables onto the base rankings table using country as the common key.

Output is 145 rows × all columns.

# Task 5: Finalizing Raw Dataset
We sort all 145 countries from A to Z and save the raw dataset.

In [ ]:
#cell 7 sort alphabetical
df = df.sort_values("country").reset_index(drop=True)
print(df["country"].head(145).to_string())

0                           Afghanistan
1                               Albania
2                               Algeria
3                                Angola
4                             Argentina
5                               Armenia
6                             Australia
7                               Austria
8                            Azerbaijan
9                               Bahrain
10                           Bangladesh
11                              Belarus
12                              Belgium
13                                Beliz
14                                Benin
15                               Bhutan
16                              Bolivia
17               Bosnia and Herzegovina
18                             Botswana
19                               Brazil
20                             Bulgaria
21                         Burkina Faso
22                             Cambodia
23                             Cameroon
24                               Canada


In [ ]:
df.to_csv("military_raw_data.csv", index=False)
files.download("military_raw_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print(list(df.columns))

['country_id', 'country', 'power_index_rank', 'power_index', 'total_population', 'total_military_manpower', 'fit_for_service', 'population_reaching_military_age_annually', 'active_personnel', 'reserve_personnel', 'paramilitary', 'total_aircraft', 'fighter_aircraft', 'attack_aircraft', 'transport_aircraft', 'trainer_aircraft', 'special_mission_aircraft', 'tanker_aircraft', 'total_military_helicopters', 'attack_helicopters', 'tanks', 'armored_fighting_vehicles', 'self_propelled_artillery', 'towed_artillery', 'rocket_projectors', 'total_naval_assets', 'total_naval_fleet_tonnage_mt', 'aircraft_carriers', 'helicopter_carriers', 'submarines', 'destroyers', 'frigates', 'corvettes', 'coastal_patrol_craft', 'mine_warfare_craft', 'defense_budget_usd', 'external_debt_usd', 'gdp_usd', 'foreign_exchange_and_gold_reserves_usd', 'total_serviceable_airports', 'labour_force', 'major_ports_and_terminals', 'total_merchant_marine_fleet', 'oil_production_bbl', 'oil_consumption_bbl', 'natural_gas_production

sort_values("country") reorders rows alphabetically.

reset_index(drop=True) renumbers the rows cleanly from 0. This file is your raw data before any cleaning.

Extracting relevant metrices

We keep only the 9 columns needed for analysis and rename three of them for consistency.

In [ ]:
# Extract only the columns we need
cols = [
    'country',
    'power_index_rank',
    'active_personnel',
    'tanks',
    'total_aircraft',
    'fighter_aircraft',
    'total_naval_assets',
    'defense_budget_usd',
    'gdp_usd'
]

df_selected = df[cols]

print(f"Shape: {df_selected.shape}")
print(df_selected.head(10).to_string())

Shape: (145, 9)
       country  power_index_rank  active_personnel   tanks  total_aircraft  fighter_aircraft  total_naval_assets  defense_budget_usd       gdp_usd
0  Afghanistan               121           75000.0     0.0             5.0               0.0                 0.0        1.450000e+08  8.223800e+10
1      Albania                77            7500.0     0.0            20.0               0.0                33.0        7.200372e+08  5.136000e+10
2      Algeria                27          130000.0  1485.0           620.0             111.0               111.0        2.500000e+10  7.229120e+11
3       Angola                59          107000.0   319.0           278.0              67.0                32.0        3.120000e+10  2.782390e+11
4    Argentina                32          108000.0   362.0           241.0              23.0                42.0        9.939198e+08  1.213000e+12
5      Armenia               101           65000.0   109.0            69.0               4.0          

In [ ]:
df.rename(columns={
    'total_military_aircraft': 'total_aircraft',
    'purchasing_power_parity_usd': 'gdp_usd',
    'total_naval_fleet': 'total_naval_assets'
}, inplace=True)

In [ ]:
df_selected.to_csv("military_selected.csv", index=False)
from google.colab import files
files.download("military_selected.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 df[cols] selects only the listed columns.

 rename() standardises column names so they match exactly what we selected.

 Output is saved as military_selected.csv

# Task 6: Data Cleaning

In [ ]:
import pandas as pd
import numpy as np

We load the selected dataset and print it to get a full view of it.

In [ ]:
# Load the file

df = pd.read_csv("military_selected.csv")

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(df.to_string())


Check whether the shape is 145 rows and 9 columns.

 145 is the total number of countries on GFP, not a real military value.

 Wherever it appears in metric columns it's a scraping error. We replace it with NaN (blank) so it doesn't affect any calculations.

In [ ]:
# Fix the spurious 145 placeholder

print("BEFORE — rows where any metric column equals 145:")
metric_cols = ['active_personnel', 'tanks', 'total_aircraft',
               'fighter_aircraft', 'total_naval_assets',
               'defense_budget_usd', 'gdp_usd']

for col in metric_cols:
    mask = df[col] == 145.0
    if mask.any():
        print(f"  ⚠️  {col}: {df.loc[mask, 'country'].tolist()} → value is 145")
        df.loc[mask, col] = np.nan   # replace with NaN

print("\nAFTER — checking again:")
found = False
for col in metric_cols:
    if (df[col] == 145.0).any():
        print(f"  ⚠️  {col} still has 145")
        found = True
if not found:
    print("  ✅  No spurious 145 values remain.")

Tunisia's tank count was affected. After this step no metric column should contain 145 as a data value.

We correct known misspellings in the country column.

In [ ]:
#  Fix known country name typos

name_fixes = {
    "Beliz"  : "Belize",
    "Turkiye": "Turkey",
}

for wrong, correct in name_fixes.items():
    mask = df["country"] == wrong
    if mask.any():
        df.loc[mask, "country"] = correct
        print(f"  Fixed: '{wrong}' → '{correct}'")

print("✅  Country name fixes applied.")

"Beliz" is fixed to "Belize" and "Turkiye" to "Turkey" so names are consistent for joining and display.

We fill missing hardware counts with 0 and missing financial values with the column median.

In [ ]:
# Handle missing values
# Strategy:
#   • Hardware counts (tanks, aircraft, naval) → fill NaN with 0
#     Reason: if a country has no data it likely has none/very few.
#   • Budget & GDP → fill NaN with the column median
#     Reason: we can't assume $0 — use a neutral estimate instead.

print(f"\nMissing values BEFORE:")
print(df.isnull().sum().to_string())

# Fill hardware columns with 0
hardware_cols = ['active_personnel', 'tanks',
                 'total_aircraft', 'fighter_aircraft',
                 'total_naval_assets']
for col in hardware_cols:
    n = df[col].isnull().sum()
    if n > 0:
        df[col] = df[col].fillna(0)
        print(f"  Filled {n} NaN → 0  in '{col}'")

# Fill financial columns with median
finance_cols = ['defense_budget_usd', 'gdp_usd']
for col in finance_cols:
    n = df[col].isnull().sum()
    if n > 0:
        median = df[col].median()
        df[col] = df[col].fillna(median)
        print(f"  Filled {n} NaN → median ({median:,.0f})  in '{col}'")

fillna(0) is used for hardware counts like tanks and aircraft, no data likely means none.

fillna(median) is used for budget and GDP, therefore, we can't assume $0, so we use the middle value of all other countries as a neutral estimate.

We sort alphabetically, run a 5-point checklist, and confirm the data is clean.

In [ ]:
# Sort alphabetically & reset index

df = df.sort_values("country").reset_index(drop=True)
print(f"✅  Sorted alphabetically. First 5 countries:")
print(df["country"].head().to_list())


# Final validation check

# 1. Row count
print(f"\n  Countries  : {len(df)}  (expected 145)")

# 2. No missing values
total_missing = df.isnull().sum().sum()
print(f"  Missing    : {total_missing}  (expected 0)")

# 3. No spurious 145 in metric cols
bad_145 = sum((df[c] == 145.0).sum() for c in metric_cols)
print(f"  Bad 145s   : {bad_145}  (expected 0)")

# 4. power_index_rank goes 1 → 145
print(f"  Rank range : {df['power_index_rank'].min():.0f} → {df['power_index_rank'].max():.0f}  (expected 1 → 145)")

# 5. No duplicate countries
dupes = df["country"].duplicated().sum()
print(f"  Duplicates : {dupes}  (expected 0)")

all_good = (len(df) == 145 and total_missing == 0
            and bad_145 == 0 and dupes == 0)
print(f"\n  {'🎉 ALL GOOD — ready to export!' if all_good else '⚠️  Fix issues above before exporting.'}")

Checks confirm 145 countries, 0 missing values, no bad 145s, rank range 1→145, and no duplicate country names. Output is military_selected_clean.csv.

We save it as the cleaned file.

In [ ]:
# Preview & export

print("\nFinal dataset preview:")
print(df.head(10).to_string(index=False))

print(f"\nColumns in final dataset:")
for col in df.columns:
    print(f"  • {col}")

# Save
df.to_csv("military_selected_clean.csv", index=False)
print("\n✅  Saved as military_selected_clean.csv")

from google.colab import files
files.download("military_selected_clean.csv")

Shape: 145 rows × 9 columns
                              country  power_index_rank  active_personnel   tanks  total_aircraft  fighter_aircraft  total_naval_assets  defense_budget_usd       gdp_usd
0                         Afghanistan               121           75000.0     0.0             5.0               0.0                 0.0        1.450000e+08  8.223800e+10
1                             Albania                77            7500.0     0.0            20.0               0.0                33.0        7.200372e+08  5.136000e+10
2                             Algeria                27          130000.0  1485.0           620.0             111.0               111.0        2.500000e+10  7.229120e+11
3                              Angola                59          107000.0   319.0           278.0              67.0                32.0        3.120000e+10  2.782390e+11
4                           Argentina                32          108000.0   362.0           241.0              23.0       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

military_selected_clean.csv is obtained.